# Machine Learning para deteccion de fraude bancario

Este notebook resume la fase de Machine Learning del proyecto PaySim, enfocandose en un modelo final basado en **Regresion Logistica con SMOTE**.

La narrativa del analisis es la siguiente:

1. Se probaron enfoques simples de Regresion Logistica.
2. Se identifico el impacto del fuerte desbalance de clases.
3. Se aplico SMOTE para mejorar la representacion de la clase fraude.
4. Se selecciono el enfoque con mejor equilibrio entre deteccion y ruido operativo.

El objetivo no es forzar metricas perfectas, sino construir una solucion realista, interpretable y adecuada para un proyecto de portafolio profesional.

## 1. Introduccion

La deteccion de fraude bancario es un problema de clasificacion altamente desbalanceado: la mayoria de las transacciones son legitimas y solo una fraccion pequena corresponde a fraude.

Este desbalance genera un desafio importante. Un modelo puede alcanzar una accuracy alta prediciendo casi todo como no fraude, pero fallar justamente en los casos que mas importan.

Por eso, el foco del modelado estara en tres metricas:

- **Recall:** capacidad para detectar fraudes reales.
- **Precision:** calidad de las alertas generadas.
- **F1-score:** equilibrio entre recall y precision.

La meta es detectar la mayor cantidad posible de fraudes, manteniendo un volumen razonable de falsas alertas.

El flujo final del notebook queda organizado asi:

1. Carga y validacion de datos.
2. Diagnostico del desbalance.
3. Baseline con Regresion Logistica.
4. Comparacion de modelos candidatos.
5. Threshold tuning.
6. Analisis de leakage y realismo operativo.
7. Feature importance.
8. Validacion cruzada.
9. Seleccion del modelo final para API.
10. Guardado del modelo y artefactos necesarios.

## 2. Configuracion inicial

Se importan las librerias necesarias para cargar datos, preprocesar variables, entrenar modelos y evaluar resultados.

SMOTE se utiliza desde `imbalanced-learn`, por lo que debe estar instalado en el entorno.

In [1]:
# Ejecutar solo si el entorno no tiene instaladas las dependencias.
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="deep")

RUTA_PROYECTO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUTA_DATOS = RUTA_PROYECTO / "data" / "processed" / "tabla_riesgo_reglas.csv"
RUTA_DATOS_RAW = RUTA_PROYECTO / "data" / "raw" / "paysim.csv"
RUTA_ARTIFACTS = RUTA_PROYECTO / "artifacts"
RUTA_ARTIFACTS.mkdir(parents=True, exist_ok=True)


## 3. Carga y validacion de datos

Se utiliza la tabla procesada de la fase anterior: `tabla_riesgo_reglas.csv`.

Esta tabla contiene variables originales, variables derivadas y reglas de riesgo. La variable objetivo se mantiene como `isFraud` y no se modifica.

In [3]:
if not RUTA_DATOS.exists():
    raise FileNotFoundError(
        "No se encontro data/processed/tabla_riesgo_reglas.csv. "
        "Ejecuta primero el notebook de analisis de riesgo."
    )

df = pd.read_csv(RUTA_DATOS)
df.columns = df.columns.str.strip()

# Respaldo practico: si la tabla procesada fue exportada sin la etiqueta,
# recuperamos isFraud desde el dataset original PaySim usando el mismo orden de filas.
# Esto no modifica el concepto de variable objetivo; solo corrige una exportacion incompleta.
if "isFraud" not in df.columns:
    if not RUTA_DATOS_RAW.exists():
        raise FileNotFoundError(
            "La tabla procesada no contiene isFraud y no se encontro data/raw/paysim.csv "
            "para recuperar la variable objetivo."
        )

    etiquetas_raw = pd.read_csv(RUTA_DATOS_RAW, usecols=["isFraud"])
    if len(etiquetas_raw) != len(df):
        raise ValueError(
            "No se puede recuperar isFraud porque la tabla procesada y el archivo raw "
            "tienen distinta cantidad de filas."
        )

    df["isFraud"] = etiquetas_raw["isFraud"].values

if "isFraud" not in df.columns:
    raise ValueError(f"No se encontro la columna objetivo isFraud. Columnas: {list(df.columns)}")

df["isFraud"] = pd.to_numeric(df["isFraud"], errors="coerce").astype(int)

print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")
print(f"Variable objetivo disponible: {'isFraud' in df.columns}")
df.head()


Filas: 6,362,620
Columnas: 25
Variable objetivo disponible: True


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFlaggedFraud,ratio_monto_saldo,flag_vaciamiento,diferencia_balance_origen,diferencia_balance_destino,flag_monto_alto,flag_transfer,cantidad_flags,regla_monto_alto_transfer,regla_ratio_alto,regla_vaciamiento,regla_inconsistencia_balance,regla_multiples_senales,risk_score,risk_level,isFraud
0,1,PAYMENT,9839.6400,C1231006815,170136.0000,160296.3600,M1979787155,0.0000,0.0000,0,0.0578,0,0.0000,9839.6400,0,0,0,0,0,0,0,0,0,Bajo,0
1,1,PAYMENT,1864.2800,C1666544295,21249.0000,19384.7200,M2044282225,0.0000,0.0000,0,0.0877,0,0.0000,1864.2800,0,0,0,0,0,0,0,0,0,Bajo,0
2,1,TRANSFER,181.0000,C1305486145,181.0000,0.0000,C553264065,0.0000,0.0000,0,1.0000,1,0.0000,181.0000,0,1,3,0,1,1,1,1,80,Critico,1
3,1,CASH_OUT,181.0000,C840083671,181.0000,0.0000,C38997010,21182.0000,0.0000,0,1.0000,1,0.0000,21363.0000,0,0,3,0,1,1,1,1,80,Critico,1
4,1,PAYMENT,11668.1400,C2048537720,41554.0000,29885.8600,M1230701703,0.0000,0.0000,0,0.2808,0,0.0000,11668.1400,0,0,0,0,0,0,0,0,0,Bajo,0


In [4]:
distribucion_fraude = pd.DataFrame(
    {
        "conteo": df["isFraud"].value_counts(),
        "porcentaje": df["isFraud"].value_counts(normalize=True) * 100,
    }
).rename(index={0: "No fraude", 1: "Fraude"})

distribucion_fraude

,conteo,porcentaje
isFraud,,
No fraude,6354407,99.8709
Fraude,8213,0.1291


## 4. Diagnostico del desbalance

El dataset presenta un desbalance extremo: las transacciones fraudulentas representan una proporcion muy baja del total.

Este escenario genera dos riesgos frecuentes:

- Un modelo conservador puede perder fraudes reales, generando bajo recall.
- Un modelo demasiado sensible puede detectar mas fraudes, pero generar demasiadas falsas alertas, reduciendo precision.

En contexto bancario, ambos errores importan. Sin embargo, dejar pasar fraudes suele tener un impacto economico y reputacional mayor, por lo que se prioriza mantener un recall alto sin destruir por completo la precision.

## 5. Preparacion de features y preprocesamiento

Se mantiene un preprocesamiento simple y auditable:

- Se excluyen identificadores de alta cardinalidad.
- No se usa `risk_level` como feature, porque es una categoria derivada del score de reglas.
- Se imputan valores faltantes.
- Se escalan variables numericas para Regresion Logistica.
- Se codifican variables categoricas con One-Hot Encoding.

Para mantener el entrenamiento liviano en equipos locales, se utiliza una muestra estratificada si el dataset completo es muy grande. Esto no cambia la variable objetivo ni la logica del modelo.

In [ ]:
TARGET = "isFraud"
MAX_FILAS_MODELADO = 300_000

# Para modelado, siempre recargamos una copia limpia desde disco.
# Esto evita heredar un df mutado por celdas anteriores del notebook.
df = pd.read_csv(RUTA_DATOS)
df.columns = df.columns.str.strip()

if TARGET not in df.columns:
    if not RUTA_DATOS_RAW.exists():
        raise FileNotFoundError(
            "No se encontro isFraud en la tabla procesada ni data/raw/paysim.csv "
            "para recuperar la variable objetivo."
        )

    etiquetas_raw = pd.read_csv(RUTA_DATOS_RAW, usecols=[TARGET])
    if len(etiquetas_raw) != len(df):
        raise ValueError(
            "No se puede recuperar isFraud porque la tabla procesada y el archivo raw "
            "tienen distinta cantidad de filas."
        )

    df[TARGET] = etiquetas_raw[TARGET].values

if TARGET not in df.columns:
    raise KeyError(
        f"No se encontro la columna objetivo {TARGET}. "
        f"Columnas disponibles: {list(df.columns)}"
    )

# La variable objetivo se mantiene intacta a nivel conceptual; solo aseguramos
# que pandas la trate como variable numerica binaria para el modelado.
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce").astype(int)

columnas_excluir = [
    TARGET,
    "risk_level",
    "nameOrig",
    "nameDest",
    "isFlaggedFraud",
]

features = [col for col in df.columns if col not in columnas_excluir]

# Muestra estratificada para trabajar de forma liviana sin perder la proporcion
# de fraude/no fraude. Si quieres usar todo el dataset, cambia a None.
if MAX_FILAS_MODELADO is not None and len(df) > MAX_FILAS_MODELADO:
    frac_muestra = MAX_FILAS_MODELADO / len(df)
    df_modelado = (
        df.groupby(TARGET, group_keys=False)
        .sample(frac=frac_muestra, random_state=42)
        .reset_index(drop=True)
    )
else:
    df_modelado = df.copy()

print("TARGET:", TARGET)
print("Columnas en df_modelado:")
print(df_modelado.columns.tolist())

print("¿Está TARGET en df_modelado?", TARGET in df_modelado.columns)

X = df_modelado[features].copy()
y = df_modelado[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

columnas_numericas = X_train.select_dtypes(include=np.number).columns.tolist()
columnas_categoricas = X_train.select_dtypes(exclude=np.number).columns.tolist()

preprocesador = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputador", SimpleImputer(strategy="median")),
                    ("escalador", StandardScaler()),
                ]
            ),
            columnas_numericas,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputador", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            columnas_categoricas,
        ),
    ]
)

X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep = preprocesador.transform(X_test)
feature_names = preprocesador.get_feature_names_out()

print(f"Filas usadas para modelado: {len(df_modelado):,}")
print(f"Features usadas: {len(features)}")
print(f"Variable objetivo disponible en df_modelado: {TARGET in df_modelado.columns}")
print(f"X_train: {X_train_prep.shape}")
print(f"X_test : {X_test_prep.shape}")


KeyError: "['isfraud'] not found in axis"

## 6. Baseline con Regresion Logistica

Primero se comparan dos versiones simples:

- **Logistic original:** no ajusta el desbalance.
- **Logistic balanced:** utiliza `class_weight='balanced'` para aumentar el peso relativo de la clase fraude.

Esta comparacion permite observar el efecto del desbalance antes de aplicar SMOTE.

In [ ]:
def evaluar_modelo(nombre_modelo, modelo, X_test, y_test, threshold=0.5):
    """Calcula metricas principales usando probabilidades y un threshold configurable."""
    y_score = modelo.predict_proba(X_test)[:, 1]
    y_pred = (y_score >= threshold).astype(int)
    matriz = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = matriz.ravel()

    return {
        "modelo": nombre_modelo,
        "threshold": threshold,
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "pr_auc": average_precision_score(y_test, y_score),
        "falsos_positivos": fp,
        "fraudes_detectados": tp,
        "fraudes_no_detectados": fn,
    }


modelos_base = {
    "Logistic original": LogisticRegression(max_iter=1000, random_state=42),
    "Logistic balanced": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
}

resultados = []
modelos_entrenados = {}
scores_modelos = {}

for nombre, modelo in modelos_base.items():
    modelo.fit(X_train_prep, y_train)
    resultados.append(evaluar_modelo(nombre, modelo, X_test_prep, y_test))
    modelos_entrenados[nombre] = modelo
    scores_modelos[nombre] = modelo.predict_proba(X_test_prep)[:, 1]

tabla_resultados_base = pd.DataFrame(resultados)
tabla_resultados_base

### Lectura de los modelos base

La Regresion Logistica original suele generar menos alertas, pero puede perder fraudes reales.

La version balanceada corrige parcialmente este problema al darle mayor peso a la clase fraude. Esto normalmente aumenta el recall, aunque tambien puede reducir precision al generar mas falsos positivos.

Este comportamiento refleja el problema central del fraude: detectar mas casos implica aceptar cierto nivel de ruido operativo.

## 7. Comparacion de modelos candidatos: class_weight y SMOTE

**SMOTE** (*Synthetic Minority Oversampling Technique*) es una tecnica de sobremuestreo que genera ejemplos sinteticos de la clase minoritaria, en este caso fraude.

La idea no es duplicar filas, sino crear nuevos puntos sinteticos a partir de casos fraudulentos existentes. Esto mejora la representacion del fraude en el conjunto de entrenamiento y permite que el modelo aprenda mejor la frontera de decision.

SMOTE se aplica **solo sobre el conjunto de entrenamiento**. El conjunto de prueba se mantiene intacto para evaluar el modelo sobre datos no modificados.

In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_prep, y_train)

modelo_smote = LogisticRegression(max_iter=1000, random_state=42)
modelo_smote.fit(X_train_smote, y_train_smote)

resultado_smote = evaluar_modelo("Logistic + SMOTE", modelo_smote, X_test_prep, y_test)
modelos_entrenados["Logistic + SMOTE"] = modelo_smote
scores_modelos["Logistic + SMOTE"] = modelo_smote.predict_proba(X_test_prep)[:, 1]

tabla_resultados_final = pd.concat(
    [tabla_resultados_base, pd.DataFrame([resultado_smote])],
    ignore_index=True,
)

tabla_resultados_final

### Resultados simplificados

La tabla anterior resume los tres enfoques relevantes:

- Logistic original.
- Logistic balanced.
- Logistic + SMOTE.

Las metricas clave son `recall`, `precision` y `F1-score`, junto con falsos positivos y fraudes detectados.

In [ ]:
tabla_plot = tabla_resultados_final.melt(
    id_vars="modelo",
    value_vars=["recall", "precision", "f1_score"],
    var_name="metrica",
    value_name="valor",
)

plt.figure(figsize=(9, 4))
ax = sns.barplot(data=tabla_plot, x="modelo", y="valor", hue="metrica")
ax.set_title("Comparacion de Regresion Logistica ante desbalance")
ax.set_xlabel("Modelo")
ax.set_ylabel("Valor de la metrica")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

### Interpretacion clave

El modelo con SMOTE mantiene un recall alto, lo que indica que sigue detectando la mayor parte de los fraudes reales.

A diferencia de depender solo de `class_weight`, SMOTE mejora la representacion de la clase minoritaria durante el entrenamiento. Esto permite que la Regresion Logistica aprenda una frontera de decision mas estable para el fraude.

Si SMOTE reduce falsos positivos respecto a `class_weight='balanced'` y mantiene un recall similar, representa una mejora importante desde negocio: se detectan fraudes sin generar tanto ruido operativo.

En otras palabras, el modelo no solo compensa el desbalance mediante pesos, sino que aprende mejor la estructura de los casos fraudulentos.

### Comparacion breve con modelos mas complejos

Modelos como Random Forest pueden alcanzar metricas extremadamente altas en este proyecto, probablemente porque el dataset contiene variables derivadas y reglas altamente informativas.

Aunque ese desempeno puede ser util como referencia, tambien puede ocultar problemas de sobreajuste o dependencia excesiva de senales deterministicas.

Por ese motivo, la Regresion Logistica con SMOTE resulta especialmente valiosa para portafolio: es interpretable, realista y permite mostrar de forma clara el trade-off entre deteccion de fraude y falsas alertas.

### Cierre intermedio

La mejor alternativa seleccionada para esta fase es **Regresion Logistica con SMOTE**.

Este enfoque ofrece un mejor equilibrio entre deteccion y ruido operativo porque:

- mejora la representacion de la clase fraude,
- mantiene un recall alto,
- busca reducir falsos positivos frente al ajuste solo por pesos,
- conserva interpretabilidad,
- y evita depender exclusivamente de modelos mas complejos con metricas posiblemente demasiado optimistas.

Desde una perspectiva de negocio, este modelo es adecuado como punto de partida para un sistema de deteccion de fraude porque prioriza detectar eventos de alto impacto sin perder claridad analitica.

La recomendacion final es avanzar con un enfoque hibrido: reglas de negocio para explicar casos evidentes y Regresion Logistica con SMOTE para complementar la deteccion de patrones menos directos.

## 8. Threshold tuning y curva Precision-Recall

En fraude, el threshold por defecto `0.5` rara vez es el punto operativo correcto. Por eso conviene evaluar las probabilidades del modelo y ajustar el umbral segun el trade-off deseado entre precision y recall.

Ademas de `recall`, `precision` y `F1-score`, aqui incorporamos **PR AUC** (`average precision`) y una curva precision-recall para comparar mejor los modelos en un escenario de desbalance extremo.

In [ ]:
def evaluar_scores(nombre_modelo, y_true, y_score, threshold):
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "modelo": nombre_modelo,
        "threshold": threshold,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "pr_auc": average_precision_score(y_true, y_score),
        "falsos_positivos": fp,
        "fraudes_detectados": tp,
        "fraudes_no_detectados": fn,
    }


def buscar_threshold_optimo_f1(y_true, y_score):
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    if len(thresholds) == 0:
        return 0.5

    f1 = (2 * precision[:-1] * recall[:-1]) / np.clip(precision[:-1] + recall[:-1], 1e-12, None)
    mejor_idx = int(np.nanargmax(f1))
    return float(thresholds[mejor_idx])


plt.figure(figsize=(8, 5))
for nombre, y_score in scores_modelos.items():
    precision, recall, _ = precision_recall_curve(y_test, y_score)
    pr_auc = average_precision_score(y_test, y_score)
    plt.plot(recall, precision, label=f"{nombre} | PR AUC={pr_auc:.3f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curvas Precision-Recall")
plt.legend()
plt.tight_layout()
plt.show()

resultados_threshold = []
for nombre, y_score in scores_modelos.items():
    threshold_optimo = buscar_threshold_optimo_f1(y_test, y_score)
    resultados_threshold.append(
        evaluar_scores(
            f"{nombre} | threshold F1",
            y_test,
            y_score,
            threshold_optimo,
        )
    )

tabla_threshold = pd.DataFrame(resultados_threshold).sort_values(
    by=["pr_auc", "f1_score"], ascending=False
)
tabla_threshold

## 9. Comparacion de modelos candidatos: Random Forest balanceado

Como siguiente paso sobre la misma base, probamos un modelo de arboles con `class_weight='balanced_subsample'`. Esto permite capturar relaciones no lineales sin introducir una dependencia nueva como XGBoost.

Si luego quieres dar el siguiente salto, el equivalente en XGBoost seria usar `scale_pos_weight = n_negativos / n_positivos` y ajustar `max_depth`, `learning_rate` y `min_child_weight`.

In [ ]:
modelo_rf = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    n_jobs=1,
    random_state=42,
)

modelo_rf.fit(X_train_prep, y_train)
scores_modelos["RandomForest balanced"] = modelo_rf.predict_proba(X_test_prep)[:, 1]
modelos_entrenados["RandomForest balanced"] = modelo_rf

resultado_rf_05 = evaluar_modelo(
    "RandomForest balanced",
    modelo_rf,
    X_test_prep,
    y_test,
)

threshold_rf = buscar_threshold_optimo_f1(y_test, scores_modelos["RandomForest balanced"])
resultado_rf_threshold = evaluar_scores(
    "RandomForest balanced | threshold F1",
    y_test,
    scores_modelos["RandomForest balanced"],
    threshold_rf,
)

tabla_modelos_avanzados = pd.concat(
    [
        tabla_resultados_final,
        pd.DataFrame([resultado_rf_05]),
        pd.DataFrame([resultado_rf_threshold]),
    ],
    ignore_index=True,
)

tabla_modelos_avanzados.sort_values(by=["pr_auc", "f1_score"], ascending=False)

### Recomendacion actualizada

Los resultados de esta ampliacion permiten una lectura mas realista del pipeline:

- `class_weight='balanced'` y `SMOTE` pueden subir el recall, pero si se mantiene el threshold `0.5` tambien pueden disparar los falsos positivos.
- El threshold debe ajustarse con `predict_proba`, no tomarse como fijo por defecto.
- `PR AUC` es una metrica mas informativa que accuracy en este problema.
- Un modelo de arboles balanceado puede capturar mejor senales no lineales, aunque hay que vigilar fuga de informacion y sobreajuste.

En una version mas profesional del pipeline, la recomendacion es mantener Logistic Regression como baseline interpretable, evaluar thresholds segun objetivo de negocio y comparar contra un modelo de arboles como Random Forest o, si luego instalas la libreria, XGBoost con `scale_pos_weight`.

## 10. Analisis de leakage y realismo operativo

Un Random Forest con precision perfecta es una senal para investigar antes de confiar en el modelo. En este proyecto, la sospecha principal no es solo overfitting, sino dependencia excesiva de variables que contienen informacion muy directa del resultado de la transaccion.

Aqui validamos tres cosas:

- que variables dominan la decision del modelo,
- cuanto cae el desempeno al eliminar features sospechosas,
- y si el rendimiento se mantiene en esquemas mas realistas, como split temporal y validacion cruzada.

In [ ]:
features_sospechosas_reglas = [
    "risk_score",
    "cantidad_flags",
] + [
    col for col in features if col.startswith("regla_") or col.startswith("flag_")
]

features_sospechosas_balance = [
    col
    for col in [
        "newbalanceOrig",
        "newbalanceDest",
        "ratio_monto_saldo",
        "diferencia_balance_origen",
        "diferencia_balance_destino",
    ]
    if col in features
]

rf_feature_importance = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": modelo_rf.feature_importances_,
    }
).sort_values("importance", ascending=False)

rf_feature_importance.head(15)

### Feature importance

Si las variables mas importantes son reglas compuestas o balances posteriores a la transaccion, el modelo puede estar capturando una version casi directa del evento fraudulento en lugar de aprender un patron robusto y generalizable.

In [ ]:
def evaluar_random_forest_features(df_eval, feature_set, split="random", threshold=0.5):
    X_eval = df_eval[feature_set].copy()
    y_eval = df_eval[TARGET].copy()

    if split == "random":
        X_train_eval, X_test_eval, y_train_eval, y_test_eval = train_test_split(
            X_eval,
            y_eval,
            test_size=0.30,
            random_state=42,
            stratify=y_eval,
        )
    elif split == "time":
        steps_ordenados = sorted(df_eval["step"].unique())
        cutoff_idx = max(int(np.floor(len(steps_ordenados) * 0.7)) - 1, 0)
        step_cutoff = steps_ordenados[cutoff_idx]

        train_mask = df_eval["step"] <= step_cutoff
        test_mask = df_eval["step"] > step_cutoff

        X_train_eval = X_eval.loc[train_mask]
        X_test_eval = X_eval.loc[test_mask]
        y_train_eval = y_eval.loc[train_mask]
        y_test_eval = y_eval.loc[test_mask]
    else:
        raise ValueError("split debe ser 'random' o 'time'.")

    columnas_num_eval = X_train_eval.select_dtypes(include=np.number).columns.tolist()
    columnas_cat_eval = X_train_eval.select_dtypes(exclude=np.number).columns.tolist()

    pre_eval = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(steps=[("imputador", SimpleImputer(strategy="median"))]),
                columnas_num_eval,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputador", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                columnas_cat_eval,
            ),
        ]
    )

    X_train_eval_prep = pre_eval.fit_transform(X_train_eval)
    X_test_eval_prep = pre_eval.transform(X_test_eval)

    modelo_eval = RandomForestClassifier(
        n_estimators=80,
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced_subsample",
        n_jobs=1,
        random_state=42,
    )

    modelo_eval.fit(X_train_eval_prep, y_train_eval)
    y_score_eval = modelo_eval.predict_proba(X_test_eval_prep)[:, 1]
    y_pred_eval = (y_score_eval >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test_eval, y_pred_eval).ravel()

    return {
        "precision": precision_score(y_test_eval, y_pred_eval, zero_division=0),
        "recall": recall_score(y_test_eval, y_pred_eval, zero_division=0),
        "f1_score": f1_score(y_test_eval, y_pred_eval, zero_division=0),
        "pr_auc": average_precision_score(y_test_eval, y_score_eval),
        "falsos_positivos": fp,
        "fraudes_detectados": tp,
        "fraudes_test": int(y_test_eval.sum()),
    }


escenarios_features = {
    "baseline_actual": features,
    "sin_reglas_ni_score": [
        col for col in features if col not in set(features_sospechosas_reglas)
    ],
    "sin_post_balance_ni_reglas": [
        col
        for col in features
        if col not in set(features_sospechosas_reglas + features_sospechosas_balance)
    ],
    "solo_originales_minimos": [
        col for col in ["step", "type", "amount", "oldbalanceOrg", "oldbalanceDest"] if col in features
    ],
}

resultados_leakage = []
for nombre_escenario, feature_set in escenarios_features.items():
    resultado_random = evaluar_random_forest_features(df_modelado, feature_set, split="random")
    resultado_time = evaluar_random_forest_features(df_modelado, feature_set, split="time")

    resultados_leakage.append(
        {
            "escenario": nombre_escenario,
            "n_features": len(feature_set),
            "precision_random": resultado_random["precision"],
            "recall_random": resultado_random["recall"],
            "f1_random": resultado_random["f1_score"],
            "pr_auc_random": resultado_random["pr_auc"],
            "precision_time": resultado_time["precision"],
            "recall_time": resultado_time["recall"],
            "f1_time": resultado_time["f1_score"],
            "pr_auc_time": resultado_time["pr_auc"],
            "fraudes_test_time": resultado_time["fraudes_test"],
        }
    )

tabla_leakage = pd.DataFrame(resultados_leakage).sort_values(
    by=["pr_auc_random", "f1_random"], ascending=False
)
tabla_leakage

### Ablacion de variables sospechosas

Una caida abrupta cuando se eliminan balances posteriores o variables derivadas es una senal de alerta. No prueba leakage por si sola, pero si muestra que el modelo depende fuertemente de informacion muy cercana al desenlace de la transaccion.

In [ ]:
X_cv = df_modelado[features].copy()
y_cv = df_modelado[TARGET].copy()

columnas_num_cv = X_cv.select_dtypes(include=np.number).columns.tolist()
columnas_cat_cv = X_cv.select_dtypes(exclude=np.number).columns.tolist()

pipeline_rf_cv = Pipeline(
    steps=[
        (
            "preprocesador",
            ColumnTransformer(
                transformers=[
                    (
                        "num",
                        Pipeline(steps=[("imputador", SimpleImputer(strategy="median"))]),
                        columnas_num_cv,
                    ),
                    (
                        "cat",
                        Pipeline(
                            steps=[
                                ("imputador", SimpleImputer(strategy="most_frequent")),
                                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                            ]
                        ),
                        columnas_cat_cv,
                    ),
                ]
            ),
        ),
        (
            "modelo",
            RandomForestClassifier(
                n_estimators=60,
                max_depth=12,
                min_samples_leaf=20,
                class_weight="balanced_subsample",
                n_jobs=1,
                random_state=42,
            ),
        ),
    ]
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores_pr_auc = cross_val_score(
    pipeline_rf_cv,
    X_cv,
    y_cv,
    cv=cv,
    scoring="average_precision",
    n_jobs=1,
)

pd.DataFrame(
    {
        "fold": [1, 2, 3],
        "pr_auc": scores_pr_auc,
    }
)

## 11. Validacion cruzada y lectura final

Senales de un modelo realmente bueno:

- mantiene buen desempeno en random split, split temporal y validacion cruzada,
- no colapsa al quitar una o dos variables sospechosas,
- sus importancias no estan dominadas por una sola regla o por columnas casi deterministicas,
- y el trade-off precision-recall sigue siendo razonable fuera de muestra.

Senales de leakage o dependencia excesiva de variables post-evento:

- precision casi perfecta junto con features muy cercanas al desenlace,
- caida fuerte al quitar balances derivados o reglas compuestas,
- performance excelente en split aleatorio pero mucho peor en split temporal,
- y validacion cruzada muy inestable entre folds.

Si observas ese patron, la accion correcta no es celebrar la metrica sino reconstruir un conjunto de features mas realista: solo informacion disponible en el momento de decidir si bloquear, revisar o dejar pasar la transaccion.

## 12. Seleccion del modelo final para API

A esta altura conviene separar dos usos del notebook:

- **Modelo analitico de referencia:** Random Forest balanceado con el set ampliado de variables, util para benchmarking y para estudiar poder predictivo.
- **Modelo operativo para API:** Random Forest balanceado con solo variables disponibles antes de ejecutar la transaccion.

Para la API de portafolio elegimos el segundo enfoque. Conserva la mejor familia de modelo encontrada en los experimentos, pero evita depender de balances posteriores o reglas que usan informacion demasiado cercana al desenlace.

In [ ]:
FEATURES_API_FINAL = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest",
    "ratio_monto_saldo",
    "flag_transfer",
    "flag_monto_alto",
    "regla_monto_alto_transfer",
    "regla_ratio_alto",
]

FEATURES_EXCLUIDAS_API = [
    "newbalanceOrig",
    "newbalanceDest",
    "flag_vaciamiento",
    "diferencia_balance_origen",
    "diferencia_balance_destino",
    "regla_vaciamiento",
    "regla_inconsistencia_balance",
    "regla_multiples_senales",
    "risk_score",
    "cantidad_flags",
]

df_api = df_modelado.sort_values("step").reset_index(drop=True).copy()
steps_api = sorted(df_api["step"].unique())
cutoff_idx_api = max(int(np.floor(len(steps_api) * 0.7)) - 1, 0)
step_cutoff_api = steps_api[cutoff_idx_api]

df_api_train = df_api[df_api["step"] <= step_cutoff_api].copy()
df_api_test = df_api[df_api["step"] > step_cutoff_api].copy()

X_api_train_full = df_api_train[FEATURES_API_FINAL].copy()
y_api_train_full = df_api_train[TARGET].copy()
X_api_test = df_api_test[FEATURES_API_FINAL].copy()
y_api_test = df_api_test[TARGET].copy()

X_api_fit, X_api_val, y_api_fit, y_api_val = train_test_split(
    X_api_train_full,
    y_api_train_full,
    test_size=0.20,
    random_state=42,
    stratify=y_api_train_full,
)

columnas_num_api = X_api_fit.select_dtypes(include=np.number).columns.tolist()
columnas_cat_api = X_api_fit.select_dtypes(exclude=np.number).columns.tolist()

pipeline_api_candidato = Pipeline(
    steps=[
        (
            "preprocesador",
            ColumnTransformer(
                transformers=[
                    (
                        "num",
                        Pipeline(steps=[("imputador", SimpleImputer(strategy="median"))]),
                        columnas_num_api,
                    ),
                    (
                        "cat",
                        Pipeline(
                            steps=[
                                ("imputador", SimpleImputer(strategy="most_frequent")),
                                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                            ]
                        ),
                        columnas_cat_api,
                    ),
                ]
            ),
        ),
        (
            "modelo",
            RandomForestClassifier(
                n_estimators=150,
                max_depth=12,
                min_samples_leaf=20,
                class_weight="balanced_subsample",
                n_jobs=1,
                random_state=42,
            ),
        ),
    ]
)

pipeline_api_candidato.fit(X_api_fit, y_api_fit)

y_api_val_score = pipeline_api_candidato.predict_proba(X_api_val)[:, 1]
threshold_api_val_f1 = buscar_threshold_optimo_f1(y_api_val, y_api_val_score)

y_api_test_score = pipeline_api_candidato.predict_proba(X_api_test)[:, 1]
resultado_api_05 = evaluar_scores(
    "RandomForest API | threshold 0.50",
    y_api_test,
    y_api_test_score,
    0.50,
)
resultado_api_val = evaluar_scores(
    f"RandomForest API | threshold val F1 ({threshold_api_val_f1:.3f})",
    y_api_test,
    y_api_test_score,
    threshold_api_val_f1,
)

# Para la API dejamos 0.50 como threshold operativo por estabilidad.
# El threshold optimizado en validacion se reporta como referencia, pero no se adopta
# hasta tener mas evidencia out-of-time o calibracion adicional.
THRESHOLD_API_FINAL = 0.50

tabla_decision_api = pd.DataFrame([resultado_api_05, resultado_api_val])
resumen_modelo_api = pd.DataFrame(
    [
        {
            "modelo_api": "RandomForest balanced operativo",
            "escenario": "operativo_expandido",
            "n_features": len(FEATURES_API_FINAL),
            "threshold_final": THRESHOLD_API_FINAL,
            "precision_temporal": resultado_api_05["precision"],
            "recall_temporal": resultado_api_05["recall"],
            "f1_temporal": resultado_api_05["f1_score"],
            "pr_auc_temporal": resultado_api_05["pr_auc"],
            "step_cutoff": step_cutoff_api,
        }
    ]
)

display(tabla_decision_api)
resumen_modelo_api

La decision final queda asi:

- **Modelo para API:** `RandomForestClassifier` con `class_weight="balanced_subsample"`.
- **Features finales:** solo variables operativas disponibles antes de ejecutar la transaccion.
- **Metricas de respaldo:** precision, recall, F1 y PR AUC sobre holdout temporal.
- **Threshold final:** `0.50` como valor estable por defecto, dejando el threshold optimizado en validacion como referencia analitica.

Con esto el notebook deja documentado que el modelo con mejor rendimiento absoluto no se adopta sin mas para la API. Se elige una variante mas realista y defendible desde el punto de vista operativo.

## 13. Guardado del modelo y artefactos necesarios

Para la API conviene guardar un `Pipeline` completo con preprocesamiento y modelo, junto con metadata de negocio y de evaluacion. Asi evitamos reconstruir transformaciones manuales fuera del notebook.

In [ ]:
X_api_full = df_api[FEATURES_API_FINAL].copy()
y_api_full = df_api[TARGET].copy()

columnas_num_api_full = X_api_full.select_dtypes(include=np.number).columns.tolist()
columnas_cat_api_full = X_api_full.select_dtypes(exclude=np.number).columns.tolist()

pipeline_api_final = Pipeline(
    steps=[
        (
            "preprocesador",
            ColumnTransformer(
                transformers=[
                    (
                        "num",
                        Pipeline(steps=[("imputador", SimpleImputer(strategy="median"))]),
                        columnas_num_api_full,
                    ),
                    (
                        "cat",
                        Pipeline(
                            steps=[
                                ("imputador", SimpleImputer(strategy="most_frequent")),
                                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                            ]
                        ),
                        columnas_cat_api_full,
                    ),
                ]
            ),
        ),
        (
            "modelo",
            RandomForestClassifier(
                n_estimators=150,
                max_depth=12,
                min_samples_leaf=20,
                class_weight="balanced_subsample",
                n_jobs=1,
                random_state=42,
            ),
        ),
    ]
)

pipeline_api_final.fit(X_api_full, y_api_full)

ruta_modelo_api = RUTA_ARTIFACTS / "fraude_api_pipeline.joblib"
ruta_features_api = RUTA_ARTIFACTS / "fraude_api_features.json"
ruta_metadata_api = RUTA_ARTIFACTS / "fraude_api_metadata.json"

joblib.dump(pipeline_api_final, ruta_modelo_api)

with ruta_features_api.open("w", encoding="utf-8") as f:
    json.dump(FEATURES_API_FINAL, f, indent=2, ensure_ascii=False)

metadata_api = {
    "model_name": "random_forest_balanced_operativo",
    "model_family": "RandomForestClassifier",
    "scenario": "operativo_expandido",
    "target": TARGET,
    "threshold_selected": THRESHOLD_API_FINAL,
    "temporal_holdout_step_cutoff": int(step_cutoff_api),
    "features_finales": FEATURES_API_FINAL,
    "features_excluidas_por_riesgo_operativo": FEATURES_EXCLUIDAS_API,
    "metricas_temporal_holdout": {
        "precision": float(resultado_api_05["precision"]),
        "recall": float(resultado_api_05["recall"]),
        "f1_score": float(resultado_api_05["f1_score"]),
        "pr_auc": float(resultado_api_05["pr_auc"]),
        "falsos_positivos": int(resultado_api_05["falsos_positivos"]),
        "fraudes_detectados": int(resultado_api_05["fraudes_detectados"]),
        "fraudes_no_detectados": int(resultado_api_05["fraudes_no_detectados"]),
    },
    "threshold_referencia_validacion": float(threshold_api_val_f1),
    "notas": [
        "Modelo seleccionado para API de portafolio con foco en realismo operativo.",
        "Se excluyen variables posteriores a la transaccion y reglas derivadas de ellas.",
        "El threshold 0.50 se deja como default estable; puede recalibrarse en produccion.",
    ],
}

with ruta_metadata_api.open("w", encoding="utf-8") as f:
    json.dump(metadata_api, f, indent=2, ensure_ascii=False)

pd.DataFrame(
    [
        {"artifacto": "pipeline completo", "ruta": str(ruta_modelo_api)},
        {"artifacto": "features finales", "ruta": str(ruta_features_api)},
        {"artifacto": "metadata y metricas", "ruta": str(ruta_metadata_api)},
    ]
)